# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
# Instalación de dependencias
!pip install -q --upgrade pypdf gradio openai python-dotenv

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [2]:
from pypdf import PdfReader

PDF_PATH = "attention_is_all_you_need.pdf"


def extract_text_from_pdf(pdf_path):
    """Lee un PDF y devuelve todo su texto como un solo string."""
    reader = PdfReader(pdf_path)
    texto_completo = ""

    for page in reader.pages:
        # Algunas páginas pueden devolver None; usamos "" para evitar errores
        texto_pagina = page.extract_text() or ""
        texto_completo = texto_completo + texto_pagina + "\n"

    return texto_completo


# Prueba de la función
texto_prueba = extract_text_from_pdf(PDF_PATH)
print(texto_prueba[:500])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Carga las variables del archivo .env
load_dotenv()

# Verificamos que la key exista SIN imprimirla
if os.getenv("NVIDIA_API_KEY"):
    print("NVIDIA_API_KEY encontrada")
else:
    raise ValueError("No se encontró NVIDIA_API_KEY. Revisa tu archivo .env")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"

NVIDIA_API_KEY encontrada


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [4]:
def build_system_prompt(document_text):
    """Construye el system prompt que incluye el paper completo."""
    prompt = (
        "Eres un asistente experto en el paper 'Attention Is All You Need' "
        "(Vaswani et al., 2017).\n\n"
        "Responde las preguntas del usuario usando ÚNICAMENTE la información "
        "del documento que aparece entre los delimitadores.\n"
        "Si la respuesta no está en el documento, responde exactamente: "
        "'No encuentro esa información en el documento'.\n"
        "No uses conocimiento externo al documento.\n\n"
        "=== INICIO DEL DOCUMENTO ===\n"
        + document_text +
        "\n=== FIN DEL DOCUMENTO ==="
    )
    return prompt


def history_to_messages(history):
    """Convierte el historial de Gradio a mensajes con roles user/assistant.

    Soporta los dos formatos que Gradio ha usado:
    - Formato nuevo: lista de diccionarios {"role": ..., "content": ...}
    - Formato antiguo: lista de pares [mensaje_usuario, respuesta_asistente]
    """
    messages = []

    for item in history:
        if isinstance(item, dict):
            # Formato nuevo
            contenido = item["content"]
            # En algunas versiones el contenido es una lista de bloques
            if isinstance(contenido, list):
                texto = ""
                for bloque in contenido:
                    if isinstance(bloque, dict) and bloque.get("type") == "text":
                        texto = texto + bloque.get("text", "")
                contenido = texto
            messages.append({"role": item["role"], "content": contenido})
        else:
            # Formato antiguo: (usuario, asistente)
            mensaje_usuario, respuesta_asistente = item
            if mensaje_usuario:
                messages.append({"role": "user", "content": mensaje_usuario})
            if respuesta_asistente:
                messages.append({"role": "assistant", "content": respuesta_asistente})

    return messages


def chat_con_documento(message, history, document_text):
    """Responde preguntas sobre el documento, entregando la respuesta por streaming."""
    # 1. Mensaje de sistema con el documento
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]

    # 2. Historial de la conversación
    messages = messages + history_to_messages(history)

    # 3. Pregunta actual
    messages.append({"role": "user", "content": message})

    # 4. Llamada al modelo con streaming
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    # 5. Acumulamos los fragmentos y los entregamos con yield
    respuesta = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        fragmento = chunk.choices[0].delta.content
        if fragmento:
            respuesta = respuesta + fragmento
            yield respuesta

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [5]:
import gradio as gr

# 1. Extraemos el texto del PDF
document_text = extract_text_from_pdf(PDF_PATH)

# 2. Mostramos cuántas páginas y caracteres se extrajeron
num_paginas = len(PdfReader(PDF_PATH).pages)
print(f"Páginas: {num_paginas}")
print(f"Caracteres extraídos: {len(document_text)}")

# 3. Preguntas de ejemplo (cada una lleva también el documento,
#    porque el Textbox oculto es una entrada adicional de la función)
preguntas_ejemplo = [
    "¿Cuál es la arquitectura principal propuesta en el paper?",
    "¿Qué es el mecanismo de atención?",
    "¿Cuántas capas tiene el encoder del modelo base?",
]
ejemplos = [[pregunta, document_text] for pregunta in preguntas_ejemplo]

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente del paper 'Attention Is All You Need'",
    description="Hazme preguntas sobre el paper. Respondo solo con base en el documento.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=ejemplos,
    cache_examples=False,
)

# 4. Lanzamos la interfaz
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)

Páginas: 15
Caracteres extraídos: 39600
* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?

Al formular dicha pregunta el modelo responde basandose solamente en el documento diciendo que no dispone de esa información, exactamente el resultado esperado.

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?

Cada pregunta manda el paper entero, y eso sale más caro (se pagan más tokens), más lento y gasta contexto. Además, hay un tope: si el documento es más largo que la ventana de contexto del modelo, simplemente no cabe. Y aunque quepa, con textos muy largos el modelo puede perderse o ignorar datos importantes.

2. ¿Por qué este ejercicio todavía no implementa RAG?

Porque aquí no se busca nada, en este ejercicio se hace algo mas simple; pegamos todo el documento en el prompt y listo. Es un primer paso que sirve para entender por qué RAG hace falta, pero no tiene fragmentos, ni búsqueda, ni selección.

3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?

 1.) Hacer preguntas trampa, como la de GPT-4. Si el modelo responde con detalles que no están en el documento entonces está usando lo que ya sabía.
2.) Pedirle que cite la página o una frase textual del documento, y comprobar a mano que esa frase existe de verdad.
3.) Cambiar el documento por uno inventado, con datos falsos, y ver si el modelo responde según el documento o según lo que recuerda.

4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

uno de los principales riesgos es que el documento podría traer instrucciones escondidas, tipo “ignora lo anterior y haz esto otro”,  chatbot puede volverse hostil y ademas comenzar a propagar desinformacion y el modelo se veria forzado obedecerlas al no poder distinguir entre estas, los documentos subidos pueden almacenarse para entrenar futuros modelos, filtrando propiedad intelectual o información personal contenida en el documento.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)